# Explore: Payments

`olist_order_payments_dataset.csv`

In [1]:
import pandas as pd

payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
payments.shape

(103886, 5)

103886 rows, 5 columns. More rows than orders (103886 vs 99441), so some orders have more than one payment row.

In [2]:
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


Columns cover the order payment data: order ID, payment method (credit card, debit card, etc.), amount paid, how many installments the payment was split into, and the sequence number of this payment within the order.

In [3]:
payments.dtypes

order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object

Order ID and payment type are `str`. Payment sequential and installments are `int64`, and payment_value is `float64` because it is a currency value.

In [4]:
payments.isnull().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

No missing values.

In [5]:
counts = payments["payment_type"].value_counts()
shares = payments["payment_type"].value_counts(normalize=True).mul(100).round(1)
payment_table = pd.DataFrame({"count": counts, "percent": shares})
payment_table

,count,percent
payment_type,,
credit_card,76795,73.9
boleto,19784,19.0
voucher,5775,5.6
debit_card,1529,1.5
not_defined,3,0.0


About 74% of payments are made using a credit card. Boleto (bank slip, transfer after order) is second at 19%, then voucher and a small amount of debit card. Only 3 rows are not_defined, negligible.

In [6]:
per_order = payments.groupby("order_id")["payment_sequential"].max()

payment_table = pd.DataFrame({
    "count": per_order.value_counts().sort_index(),
    "percent": per_order.value_counts(normalize=True).mul(100).round(1),
})
payment_table

,count,percent
payment_sequential,,
1,96401,96.9
2,2458,2.5
3,303,0.3
4,108,0.1
5,52,0.1
6,36,0.0
7,28,0.0
8,11,0.0
9,9,0.0


About 97% of orders have a single payment. 2.5% use two payments, 0.3% use three. A few orders go much higher, up to 29 payments.

In [7]:
max_seq = payments.groupby("order_id")["payment_sequential"].max()
big = max_seq[max_seq >= 10].index

payments[payments["order_id"] == big[-1]].sort_values("payment_sequential")

,order_id,payment_sequential,payment_type,payment_installments,payment_value
9665,fedcd9f7ccdc8cba3a18defedd1a5547,1,credit_card,1,1.67
27087,fedcd9f7ccdc8cba3a18defedd1a5547,2,voucher,1,7.76
87783,fedcd9f7ccdc8cba3a18defedd1a5547,3,voucher,1,26.94
45197,fedcd9f7ccdc8cba3a18defedd1a5547,4,voucher,1,10.33
36869,fedcd9f7ccdc8cba3a18defedd1a5547,5,voucher,1,9.76
103569,fedcd9f7ccdc8cba3a18defedd1a5547,6,voucher,1,8.60
95493,fedcd9f7ccdc8cba3a18defedd1a5547,7,voucher,1,11.78
15930,fedcd9f7ccdc8cba3a18defedd1a5547,8,voucher,1,31.43
14352,fedcd9f7ccdc8cba3a18defedd1a5547,9,voucher,1,5.30
44641,fedcd9f7ccdc8cba3a18defedd1a5547,10,voucher,1,13.27


Code: Take each order's highest payment_sequential to see how many payment rows it has. Orders with 10+ payments are rare; this pulls all payment rows for the last one in that list.

Result: One credit card payment followed by many small vouchers. The scattered index numbers are the CSV line numbers, so payments for one order are not stored next to each other.